# 04 — Cross-selling Opportunities

Xác định và đánh giá các cặp sản phẩm có tiềm năng bán chéo dựa trên quan hệ đồng mua và kết quả phân tích mạng.


## 1. Objective & Inputs

RQ3: How can co-purchase relationships reveal potential cross-selling opportunities?

Đọc invoice_items của Phase 1 và selected_products/pair_counts của Phase 2 theo config. Dùng mọi positive pair count trước lọc graph; MIN_EDGE_WEIGHT không phải ngưỡng association. Không cleaning, dựng graph hoặc chạy lại Louvain/Centrality. Các kết quả là bằng chứng lịch sử để sàng lọc cơ hội và thiết kế thử nghiệm, chưa chứng minh hiệu quả bán kèm.

In [1]:
from pathlib import Path
import sys

import numpy as np
import pandas as pd
from IPython.display import display

project_root = Path.cwd()
if project_root.name == "notebooks":
    project_root = project_root.parent
sys.path.insert(0, str(project_root))
from config import PROCESSED_DATA_DIR, TABLES_DIR, TOP_N_PRODUCTS, MIN_RULE_PAIR_COUNT

invoice_items = pd.read_parquet(PROCESSED_DATA_DIR / "invoice_items.parquet", columns=["InvoiceNo", "StockCode"])
products = pd.read_csv(TABLES_DIR / "selected_products.csv", dtype={"StockCode": "string"})
pairs = pd.read_csv(TABLES_DIR / "pair_counts.csv", dtype={"ProductA": "string", "ProductB": "string"})
assert type(MIN_RULE_PAIR_COUNT) is int and MIN_RULE_PAIR_COUNT > 0
N = invoice_items["InvoiceNo"].nunique()
assert N > 0


## 2. Data Validation & Association Metrics

N là số hóa đơn phân biệt trong toàn bộ cleaned invoice_items, kể cả hóa đơn không chứa sản phẩm được chọn hoặc chỉ chứa một sản phẩm. count(A) và count(B) cũng đếm hóa đơn, không dùng Quantity hay doanh thu.

- Support(A) = count(A)/N: tỷ lệ hóa đơn có A.
- Pair Support(A,B) = count(A,B)/N: độ phổ biến của cặp trên toàn bộ hóa đơn.
- Confidence(A → B) = count(A,B)/count(A): tỷ lệ hóa đơn có B trong các hóa đơn có A.
- Lift = Pair Support/(Support(A) × Support(B)): mức mua chung so với kỳ vọng độc lập. Lift > 1 tương đương Confidence(A → B) > Support(B); không tự chứng minh ý nghĩa thống kê hoặc quan hệ nhân quả.

Chỉ kiểm tra contract đầu vào cần cho công thức. Các invoice sets dưới đây được tái sử dụng cho kiểm chứng trực tiếp và large-invoice sensitivity, không đếm lại toàn bộ combinations.

In [2]:
keys = ["InvoiceNo", "StockCode"]
assert invoice_items[keys].notna().all().all() and not invoice_items.duplicated(keys).any()
assert products["StockCode"].is_unique and products[["StockCode", "Description"]].notna().all().all()
assert len(products) == TOP_N_PRODUCTS
assert pairs[["ProductA", "ProductB"]].isin(set(products["StockCode"])).all().all()
assert pairs["ProductA"].lt(pairs["ProductB"]).all() and not pairs.duplicated(["ProductA", "ProductB"]).any()
assert pd.api.types.is_integer_dtype(pairs["pair_count"]) and pairs["pair_count"].gt(0).all()
selected_items = invoice_items.loc[invoice_items["StockCode"].isin(products["StockCode"])]
invoice_sets = {code: set(invoices) for code, invoices in selected_items.groupby("StockCode")["InvoiceNo"]}
single_counts = pd.Series({code: len(invoices) for code, invoices in invoice_sets.items()})
assert products["StockCode"].map(single_counts).eq(products["invoice_count"]).all()
assert single_counts.gt(0).all() and single_counts.le(N).all()
assert pairs["pair_count"].le(np.minimum(pairs["ProductA"].map(single_counts), pairs["ProductB"].map(single_counts))).all()

pair_metrics = pairs.loc[pairs["pair_count"].ge(MIN_RULE_PAIR_COUNT)].reset_index(drop=True).rename_axis("pair_id").reset_index()
pair_metrics["count_a"] = pair_metrics["ProductA"].map(single_counts).astype("int64")
pair_metrics["count_b"] = pair_metrics["ProductB"].map(single_counts).astype("int64")
pair_metrics["support_a"] = pair_metrics["count_a"] / N
pair_metrics["support_b"] = pair_metrics["count_b"] / N
pair_metrics["pair_support"] = pair_metrics["pair_count"] / N
pair_metrics["lift"] = pair_metrics["pair_support"] / (pair_metrics["support_a"] * pair_metrics["support_b"])
display(pd.Series({"all_cleaned_invoices_N": N, "invoices_with_selected_products": selected_items["InvoiceNo"].nunique(),
    "selected_products": len(products), "positive_unordered_pairs": len(pairs),
    "MIN_RULE_PAIR_COUNT": MIN_RULE_PAIR_COUNT, "eligible_unordered_pairs": len(pair_metrics)}, name="Association input scope"))


all_cleaned_invoices_N             19773
invoices_with_selected_products    15942
selected_products                    100
positive_unordered_pairs            4950
MIN_RULE_PAIR_COUNT                    5
eligible_unordered_pairs            4950
Name: Association input scope, dtype: int64

## 3. Directed Association Rules

Mỗi cặp hợp lệ tạo đúng hai hướng. Pair Support và Lift được giữ nguyên ở hai hướng; Confidence dùng số hóa đơn của antecedent nên có thể khác nhau. StockCode là định danh, Description chỉ là nhãn diễn giải.

In [3]:
forward = pair_metrics.rename(columns={"ProductA": "antecedent", "ProductB": "consequent", "count_a": "antecedent_count", "count_b": "consequent_count", "support_a": "antecedent_support", "support_b": "consequent_support"})
reverse = pair_metrics.rename(columns={"ProductB": "antecedent", "ProductA": "consequent", "count_b": "antecedent_count", "count_a": "consequent_count", "support_b": "antecedent_support", "support_a": "consequent_support"})
rules = pd.concat([forward, reverse], ignore_index=True)
rules["confidence"] = rules["pair_count"] / rules["antecedent_count"]
descriptions = products.set_index("StockCode")["Description"]
rules["antecedent_description"] = rules["antecedent"].map(descriptions)
rules["consequent_description"] = rules["consequent"].map(descriptions)
core_columns = ["antecedent", "consequent", "antecedent_description", "consequent_description", "antecedent_count", "consequent_count", "pair_count", "pair_support", "antecedent_support", "consequent_support", "confidence", "lift"]
rules = rules[["pair_id"] + core_columns]
assert len(rules) == 2 * len(pair_metrics) and not rules.duplicated(["antecedent", "consequent"]).any()
assert rules["antecedent"].ne(rules["consequent"]).all()
assert rules.groupby("pair_id").size().eq(2).all() and rules.groupby("pair_id")[["pair_support", "lift"]].nunique().eq(1).all().all()
numeric_columns = ["antecedent_count", "consequent_count", "pair_count", "pair_support", "antecedent_support", "consequent_support", "confidence", "lift"]
assert rules[core_columns].notna().all().all() and np.isfinite(rules[numeric_columns]).all().all()
assert rules["pair_support"].gt(0).all() and rules["pair_support"].le(1).all() and rules["confidence"].between(0, 1).all() and rules["lift"].gt(0).all()
assert rules["pair_support"].le(rules[["antecedent_support", "consequent_support"]].min(axis=1)).all()
display(pd.Series({"eligible_unordered_pairs": len(pair_metrics), "directed_rules": len(rules)}, name="Rule generation"))


eligible_unordered_pairs    4950
directed_rules              9900
Name: Rule generation, dtype: int64

## 4. Rule Ranking & Candidate Selection

Ba bảng Top 10 xem xét từng góc độ: độ phổ biến của cặp, xác suất có hướng và mức vượt kỳ vọng độc lập. Khi bằng chỉ số chính, ưu tiên lần lượt các chỉ số còn lại và StockCode để kết quả tái lập.

Danh sách ứng viên sàng lọc gồm **mọi directed rule có pair_count ≥ MIN_RULE_PAIR_COUNT và Lift > 1**. Không đặt thêm ngưỡng Support/Confidence tùy ý: Confidence được đánh giá tương đối với xác suất nền Support(B), còn Support và Pair Count luôn hiển thị để xem quy mô bằng chứng. MIN_RULE_PAIR_COUNT vẫn là ngưỡng nghiên cứu ban đầu, không phải bảo đảm độ tin cậy.

Thứ tự review: Confidence giảm dần → Pair Support giảm dần → Lift giảm dần → StockCode. Confidence ưu tiên xác suất gợi ý theo hướng; Pair Support kiểm tra độ phổ biến; Lift kiểm tra mức liên hệ tương đối. Với N cố định, Pair Count và Pair Support chứa cùng thông tin xếp hạng. CSV lưu toàn bộ pool này, không chỉ Top 10; các dòng là ứng viên cần đánh giá, chưa phải đề xuất triển khai.

In [4]:
display_columns = ["antecedent", "consequent", "antecedent_description", "consequent_description", "pair_count", "pair_support", "confidence", "lift"]
top_support = rules.sort_values(["pair_support", "confidence", "lift", "antecedent", "consequent"], ascending=[False, False, False, True, True]).head(10)
top_confidence = rules.sort_values(["confidence", "pair_support", "lift", "antecedent", "consequent"], ascending=[False, False, False, True, True]).head(10)
top_lift = rules.sort_values(["lift", "pair_support", "confidence", "antecedent", "consequent"], ascending=[False, False, False, True, True]).head(10)
display(top_support[display_columns].assign(ranking_by="pair_support"))
display(top_confidence[display_columns].assign(ranking_by="confidence"))
display(top_lift[display_columns].assign(ranking_by="lift"))
candidates = rules.loc[rules["lift"].gt(1), core_columns].sort_values(
    ["confidence", "pair_support", "lift", "antecedent", "consequent"], ascending=[False, False, False, True, True]).reset_index(drop=True)
candidates.insert(0, "rank", np.arange(1, len(candidates) + 1))
candidates["total_invoices"] = N
if candidates.empty:
    display(pd.Series({"status": "No candidate meets MIN_RULE_PAIR_COUNT and Lift > 1; thresholds unchanged."}))
else:
    display(candidates.head(10)[["rank"] + display_columns])


,antecedent,consequent,antecedent_description,consequent_description,pair_count,pair_support,confidence,lift,ranking_by
3405,22386,85099B,JUMBO BAG PINK POLKADOT,JUMBO BAG RED RETROSPOT,825,0.041724,0.677340,6.411222,pair_support
8355,85099B,22386,JUMBO BAG RED RETROSPOT,JUMBO BAG PINK POLKADOT,825,0.041724,0.394926,6.411222,pair_support
4090,22697,22699,GREEN REGENCY TEACUP AND SAUCER,ROSES REGENCY TEACUP AND SAUCER,767,0.038790,0.757157,14.057525,pair_support
9040,22699,22697,ROSES REGENCY TEACUP AND SAUCER,GREEN REGENCY TEACUP AND SAUCER,767,0.038790,0.720188,14.057525,pair_support
2170,21931,85099B,JUMBO STORAGE BAG SUKI,JUMBO BAG RED RETROSPOT,724,0.036616,0.611486,5.787900,pair_support
7120,85099B,21931,JUMBO BAG RED RETROSPOT,JUMBO STORAGE BAG SUKI,724,0.036616,0.346577,5.787900,pair_support
3460,22411,85099B,JUMBO SHOPPER VINTAGE RED PAISLEY,JUMBO BAG RED RETROSPOT,680,0.034390,0.578723,5.477787,pair_support
8410,85099B,22411,JUMBO BAG RED RETROSPOT,JUMBO SHOPPER VINTAGE RED PAISLEY,680,0.034390,0.325515,5.477787,pair_support
5469,22383,20725,LUNCH BAG SUKI DESIGN,LUNCH BAG RED RETROSPOT,655,0.033126,0.510125,6.445172,pair_support
519,20725,22383,LUNCH BAG RED RETROSPOT,LUNCH BAG SUKI DESIGN,655,0.033126,0.418530,6.445172,pair_support


,antecedent,consequent,antecedent_description,consequent_description,pair_count,pair_support,confidence,lift,ranking_by
9039,22698,22697,PINK REGENCY TEACUP AND SAUCER,GREEN REGENCY TEACUP AND SAUCER,632,0.031963,0.826144,16.125707,confidence
4130,22698,22699,PINK REGENCY TEACUP AND SAUCER,ROSES REGENCY TEACUP AND SAUCER,598,0.030243,0.781699,14.513184,confidence
4090,22697,22699,GREEN REGENCY TEACUP AND SAUCER,ROSES REGENCY TEACUP AND SAUCER,767,0.038790,0.757157,14.057525,confidence
4779,23300,23301,GARDENERS KNEELING PAD CUP OF TEA,GARDENERS KNEELING PAD KEEP CALM,546,0.027613,0.720317,15.600023,confidence
9040,22699,22697,ROSES REGENCY TEACUP AND SAUCER,GREEN REGENCY TEACUP AND SAUCER,767,0.038790,0.720188,14.057525,confidence
5372,22356,20724,CHARLOTTE BAG PINK POLKADOT,RED RETROSPOT CHARLOTTE BAG,522,0.026400,0.703504,13.479056,confidence
3405,22386,85099B,JUMBO BAG PINK POLKADOT,JUMBO BAG RED RETROSPOT,825,0.041724,0.677340,6.411222,confidence
7310,22910,22086,PAPER CHAIN KIT VINTAGE CHRISTMAS,PAPER CHAIN KIT 50'S CHRISTMAS,552,0.027917,0.673171,11.474659,confidence
294,20723,20724,STRAWBERRY CHARLOTTE BAG,RED RETROSPOT CHARLOTTE BAG,486,0.024579,0.673130,12.897096,confidence
1942,21928,85099B,JUMBO BAG SCANDINAVIAN BLUE PAISLEY,JUMBO BAG RED RETROSPOT,541,0.027361,0.671216,6.353256,confidence


,antecedent,consequent,antecedent_description,consequent_description,pair_count,pair_support,confidence,lift,ranking_by
9039,22698,22697,PINK REGENCY TEACUP AND SAUCER,GREEN REGENCY TEACUP AND SAUCER,632,0.031963,0.826144,16.125707,lift
4089,22697,22698,GREEN REGENCY TEACUP AND SAUCER,PINK REGENCY TEACUP AND SAUCER,632,0.031963,0.623889,16.125707,lift
4779,23300,23301,GARDENERS KNEELING PAD CUP OF TEA,GARDENERS KNEELING PAD KEEP CALM,546,0.027613,0.720317,15.600023,lift
9729,23301,23300,GARDENERS KNEELING PAD KEEP CALM,GARDENERS KNEELING PAD CUP OF TEA,546,0.027613,0.598028,15.600023,lift
4130,22698,22699,PINK REGENCY TEACUP AND SAUCER,ROSES REGENCY TEACUP AND SAUCER,598,0.030243,0.781699,14.513184,lift
9080,22699,22698,ROSES REGENCY TEACUP AND SAUCER,PINK REGENCY TEACUP AND SAUCER,598,0.030243,0.561502,14.513184,lift
4090,22697,22699,GREEN REGENCY TEACUP AND SAUCER,ROSES REGENCY TEACUP AND SAUCER,767,0.038790,0.757157,14.057525,lift
9040,22699,22697,ROSES REGENCY TEACUP AND SAUCER,GREEN REGENCY TEACUP AND SAUCER,767,0.038790,0.720188,14.057525,lift
327,20723,22356,STRAWBERRY CHARLOTTE BAG,CHARLOTTE BAG PINK POLKADOT,379,0.019168,0.524931,13.988485,lift
5277,22356,20723,CHARLOTTE BAG PINK POLKADOT,STRAWBERRY CHARLOTTE BAG,379,0.019168,0.510782,13.988485,lift


,rank,antecedent,consequent,antecedent_description,consequent_description,pair_count,pair_support,confidence,lift
0,1,22698,22697,PINK REGENCY TEACUP AND SAUCER,GREEN REGENCY TEACUP AND SAUCER,632,0.031963,0.826144,16.125707
1,2,22698,22699,PINK REGENCY TEACUP AND SAUCER,ROSES REGENCY TEACUP AND SAUCER,598,0.030243,0.781699,14.513184
2,3,22697,22699,GREEN REGENCY TEACUP AND SAUCER,ROSES REGENCY TEACUP AND SAUCER,767,0.038790,0.757157,14.057525
3,4,23300,23301,GARDENERS KNEELING PAD CUP OF TEA,GARDENERS KNEELING PAD KEEP CALM,546,0.027613,0.720317,15.600023
4,5,22699,22697,ROSES REGENCY TEACUP AND SAUCER,GREEN REGENCY TEACUP AND SAUCER,767,0.038790,0.720188,14.057525
5,6,22356,20724,CHARLOTTE BAG PINK POLKADOT,RED RETROSPOT CHARLOTTE BAG,522,0.026400,0.703504,13.479056
6,7,22386,85099B,JUMBO BAG PINK POLKADOT,JUMBO BAG RED RETROSPOT,825,0.041724,0.677340,6.411222
7,8,22910,22086,PAPER CHAIN KIT VINTAGE CHRISTMAS,PAPER CHAIN KIT 50'S CHRISTMAS,552,0.027917,0.673171,11.474659
8,9,20723,20724,STRAWBERRY CHARLOTTE BAG,RED RETROSPOT CHARLOTTE BAG,486,0.024579,0.673130,12.897096
9,10,21928,85099B,JUMBO BAG SCANDINAVIAN BLUE PAISLEY,JUMBO BAG RED RETROSPOT,541,0.027361,0.671216,6.353256


## 5. Community-based Interpretation

Chỉ bổ sung community_id nếu output Phase 3 có cấu hình Top 100 / 125 và đúng tập StockCode đang phân tích. Không có file hoặc không khớp thì bỏ qua, không ngắt association analysis.

So sánh các ứng viên cùng và khác community; giữ cả hai nhóm trong pool. Số rules đếm theo hướng, các metrics ở hai hướng của một cặp không độc lập. Community cung cấp bối cảnh cấu trúc mạng, không xác nhận tính phù hợp để bán kèm.

Community và association rules cùng xuất phát từ dữ liệu mua chung, nên sự phù hợp giữa chúng không phải xác nhận độc lập. Số rules giữa các community còn phụ thuộc số cặp sản phẩm khả dụng; không dùng riêng số lượng rules để kết luận nhóm nào có cơ hội kinh doanh tốt hơn.

In [5]:
community_path = TABLES_DIR / "node_metrics.csv"
community_available = False
if community_path.is_file():
    nodes = pd.read_csv(community_path, dtype={"StockCode": "string"})
    needed = {"StockCode", "community_id", "top_n", "threshold"}
    community_available = (needed.issubset(nodes.columns) and not nodes.empty and nodes["StockCode"].is_unique
        and nodes["community_id"].notna().all() and nodes["top_n"].eq(100).all() and nodes["threshold"].eq(125).all()
        and set(nodes["StockCode"]) == set(products["StockCode"]))
if community_available:
    community_ids = nodes.set_index("StockCode")["community_id"]
    candidates["antecedent_community"] = candidates["antecedent"].map(community_ids)
    candidates["consequent_community"] = candidates["consequent"].map(community_ids)
    candidates["same_community"] = candidates["antecedent_community"].eq(candidates["consequent_community"])
    display(candidates.groupby("same_community").agg(directed_candidates=("rank", "size"), median_pair_support=("pair_support", "median"), median_confidence=("confidence", "median"), median_lift=("lift", "median")))
    display(candidates.groupby("same_community", sort=False).head(5)[["rank"] + display_columns + ["antecedent_community", "consequent_community", "same_community"]])
else:
    display(pd.Series({"community_integration": "skipped", "reason": "Phase 3 output missing or does not match Top 100/125 and selected products."}))


,directed_candidates,median_pair_support,median_confidence,median_lift
same_community,,,,
False,7042,0.005260,0.111861,2.377038
True,2752,0.007434,0.148691,3.042549


,rank,antecedent,consequent,antecedent_description,consequent_description,pair_count,pair_support,confidence,lift,antecedent_community,consequent_community,same_community
0,1,22698,22697,PINK REGENCY TEACUP AND SAUCER,GREEN REGENCY TEACUP AND SAUCER,632,0.031963,0.826144,16.125707,3,3,True
1,2,22698,22699,PINK REGENCY TEACUP AND SAUCER,ROSES REGENCY TEACUP AND SAUCER,598,0.030243,0.781699,14.513184,3,3,True
2,3,22697,22699,GREEN REGENCY TEACUP AND SAUCER,ROSES REGENCY TEACUP AND SAUCER,767,0.038790,0.757157,14.057525,3,3,True
3,4,23300,23301,GARDENERS KNEELING PAD CUP OF TEA,GARDENERS KNEELING PAD KEEP CALM,546,0.027613,0.720317,15.600023,5,5,True
4,5,22699,22697,ROSES REGENCY TEACUP AND SAUCER,GREEN REGENCY TEACUP AND SAUCER,767,0.038790,0.720188,14.057525,3,3,True
88,89,23209,23203,LUNCH BAG VINTAGE DOILY,JUMBO BAG VINTAGE DOILY,504,0.025489,0.459016,7.372974,2,1,False
143,144,20724,85099B,RED RETROSPOT CHARLOTTE BAG,JUMBO BAG RED RETROSPOT,425,0.021494,0.411822,3.898014,2,1,False
149,150,23203,23209,JUMBO BAG VINTAGE DOILY,LUNCH BAG VINTAGE DOILY,504,0.025489,0.409423,7.372974,1,2,False
186,187,22356,85099B,CHARLOTTE BAG PINK POLKADOT,JUMBO BAG RED RETROSPOT,280,0.014161,0.377358,3.571809,2,1,False
190,191,23199,23206,JUMBO BAG APPLES,LUNCH BAG APPLE DESIGN,365,0.018460,0.373211,7.123064,1,2,False


## 6. Large Invoice Sensitivity

Xác định hóa đơn có ≥100 StockCode phân biệt từ **toàn bộ basket trước chọn Top N**. Với 20 directed candidates đứng đầu, đếm trực tiếp giao của các invoice sets với tập large invoices; hai hướng dùng cùng pair count contribution. Không duyệt combinations hoặc tự loại hóa đơn lớn khỏi baseline.

large_invoice_pair_share là tỷ lệ Pair Count đến từ large invoices (0–1). Xếp theo tỷ lệ đóng góp giúp xác định các rules cần kiểm tra thêm mà không đặt nhãn cao/thấp bằng ngưỡng tùy ý. Đây là sensitivity của mẫu 20 rules được review, không đại diện toàn bộ pool; không tính lại Support/Confidence/Lift cho scenario khác trên mẫu số baseline.

In [6]:
basket_sizes = invoice_items.groupby("InvoiceNo")["StockCode"].nunique()
large_invoice_ids = set(basket_sizes.index[basket_sizes.ge(100)])
review_candidates = candidates.head(20).copy()
review_keys = np.sort(review_candidates[["antecedent", "consequent"]].to_numpy(), axis=1)
large_pair_counts = {(a, b): len(invoice_sets[a] & invoice_sets[b] & large_invoice_ids) for a, b in set(map(tuple, review_keys))}
review_candidates["large_invoice_pair_count"] = [large_pair_counts[(a, b)] for a, b in review_keys]
review_candidates["large_invoice_pair_share"] = review_candidates["large_invoice_pair_count"] / review_candidates["pair_count"]
assert review_candidates["large_invoice_pair_count"].between(0, review_candidates["pair_count"]).all()
assert review_candidates["large_invoice_pair_share"].between(0, 1).all()
display(pd.Series({"all_invoices": N, "large_invoices": len(large_invoice_ids), "large_invoice_fraction": len(large_invoice_ids) / N,
    "reviewed_directed_candidates": len(review_candidates), "reviewed_unordered_pairs": len(large_pair_counts)}, name="Sensitivity scope"))
display(review_candidates.sort_values(["large_invoice_pair_share", "rank"], ascending=[False, True])[
    ["rank", "antecedent", "consequent", "pair_count", "confidence", "lift", "large_invoice_pair_count", "large_invoice_pair_share"]])


all_invoices                    19773.000000
large_invoices                    636.000000
large_invoice_fraction              0.032165
reviewed_directed_candidates       20.000000
reviewed_unordered_pairs           17.000000
Name: Sensitivity scope, dtype: float64

,rank,antecedent,consequent,pair_count,confidence,lift,large_invoice_pair_count,large_invoice_pair_share
5,6,22356,20724,522,0.703504,13.479056,157,0.300766
8,9,20723,20724,486,0.673130,12.897096,142,0.292181
17,18,20712,85099B,535,0.619930,5.867824,155,0.289720
2,3,22697,22699,767,0.757157,14.057525,217,0.282920
4,5,22699,22697,767,0.720188,14.057525,217,0.282920
12,13,85099F,85099B,536,0.652068,6.172017,141,0.263060
1,2,22698,22699,598,0.781699,14.513184,157,0.262542
9,10,21928,85099B,541,0.671216,6.353256,142,0.262477
0,1,22698,22697,632,0.826144,16.125707,164,0.259494
16,17,22697,22698,632,0.623889,16.125707,164,0.259494


### Cross-selling Shortlist — P75, Pareto và diversity

Sàng lọc directed candidates theo P75 riêng của Pair Support, Confidence và Lift; phải đạt cả ba. P75 là heuristic khám phá, chưa phải ngưỡng tối ưu đã kiểm chứng. Giữ nguyên candidate pool và MIN_RULE_PAIR_COUNT.

A dominates B khi không thấp hơn trên cả ba metrics và cao hơn ở ít nhất một metric. Front 1 không bị dominates; bỏ front này rồi xác định các front tiếp theo. Ties hoàn toàn nằm cùng front. Chỉ tính ma trận dominance trên tập đã sàng lọc, với bộ nhớ O(m²), không trên toàn bộ pool; không cộng điểm hay gán trọng số.

Các giới hạn khai báo tập trung: tối đa 20 rules, mỗi sản phẩm xuất hiện tối đa 2 lần tính chung antecedent/consequent, mỗi cặp community không phân biệt thứ tự tối đa 4 rules nếu có metadata hợp lệ. Đây là giới hạn khám phá nhằm giảm lặp sản phẩm và tập trung community, không phải quota hay cấu hình tối ưu.

Mỗi unordered product pair giữ hướng Confidence cao hơn; hòa thì StockCode tăng dần. Duyệt theo front tăng dần, rồi Pair Support → Confidence → Lift giảm dần và StockCode tăng dần; bỏ rule vượt giới hạn. Cùng/khác community đều được xét, không dành suất riêng cho cross-community và không nới P75 nếu thiếu rules.

Business review diễn ra sau lựa chọn: không phân loại ngữ nghĩa tự động từ tên, nên relationship_type để unclear và needs_business_review = True. Các quan hệ vẫn cần xác nhận tính phù hợp trước khi thử nghiệm kinh doanh.

In [7]:
MAX_SHORTLIST_RULES = 20
MAX_PRODUCT_APPEARANCES = 2
MAX_COMMUNITY_PAIR_RULES = 4
screen_metrics = ["pair_support", "confidence", "lift"]
shortlist_bounds = candidates[screen_metrics].quantile(0.75)
screened = candidates.loc[candidates[screen_metrics].ge(shortlist_bounds).all(axis=1)].copy()
display(shortlist_bounds.rename("P75 screening boundary").to_frame())

values = screened[screen_metrics].to_numpy()
dominates = np.ones((len(screened), len(screened)), dtype=bool)
strictly_better = np.zeros_like(dominates)
for metric in values.T:
    dominates &= metric[:, None] >= metric[None, :]
    strictly_better |= metric[:, None] > metric[None, :]
dominates &= strictly_better
fronts = np.zeros(len(screened), dtype=int)
domination_counts = dominates.sum(axis=0)
front = 1
while (fronts == 0).any():
    current_front = np.flatnonzero((fronts == 0) & (domination_counts == 0))
    assert len(current_front) > 0
    fronts[current_front] = front
    domination_counts -= dominates[current_front].sum(axis=0)
    front += 1
if len(screened):
    assert np.array_equal(fronts, 1 + (dominates * fronts[:, None]).max(axis=0))
screened["pareto_front"] = fronts
del dominates, strictly_better

canonical_pairs = np.sort(screened[["antecedent", "consequent"]].to_numpy(), axis=1)
best_directions = screened.assign(product_a=canonical_pairs[:, 0], product_b=canonical_pairs[:, 1]).sort_values(
    ["confidence", "antecedent", "consequent"], ascending=[False, True, True]).drop_duplicates(["product_a", "product_b"])
best_directions = best_directions.sort_values(
    ["pareto_front", "pair_support", "confidence", "lift", "antecedent", "consequent"],
    ascending=[True, False, False, False, True, True])
product_uses, community_pair_uses, selected_indices = {}, {}, []
for index, row in best_directions.iterrows():
    a, b = row["antecedent"], row["consequent"]
    if product_uses.get(a, 0) >= MAX_PRODUCT_APPEARANCES or product_uses.get(b, 0) >= MAX_PRODUCT_APPEARANCES:
        continue
    if community_available:
        community_pair = tuple(sorted([row["antecedent_community"], row["consequent_community"]]))
        if community_pair_uses.get(community_pair, 0) >= MAX_COMMUNITY_PAIR_RULES:
            continue
        community_pair_uses[community_pair] = community_pair_uses.get(community_pair, 0) + 1
    selected_indices.append(index)
    product_uses[a] = product_uses.get(a, 0) + 1
    product_uses[b] = product_uses.get(b, 0) + 1
    if len(selected_indices) == MAX_SHORTLIST_RULES:
        break
shortlist = screened.loc[selected_indices].reset_index(drop=True)
shortlist.insert(0, "shortlist_rank", np.arange(1, len(shortlist) + 1))
for column in ["antecedent_community", "consequent_community"]:
    if column not in shortlist:
        shortlist[column] = pd.Series(pd.NA, index=shortlist.index, dtype="Int64")
if "same_community" not in shortlist:
    shortlist["same_community"] = pd.Series(pd.NA, index=shortlist.index, dtype="boolean")
shortlist["relationship_type"] = "unclear"
shortlist["needs_business_review"] = True
community_reason = f"cặp community ≤ {MAX_COMMUNITY_PAIR_RULES}" if community_available else "chưa có community hợp lệ"
shortlist["selection_reason"] = [
    f"Front {f}; support={s:.2%}, confidence={c:.2%}, lift={l:.2f} đạt P75; mỗi sản phẩm ≤ {MAX_PRODUCT_APPEARANCES}, {community_reason}."
    for f, s, c, l in shortlist[["pareto_front"] + screen_metrics].itertuples(index=False, name=None)
]
display(screened.groupby("pareto_front").size().rename("screened_rules").to_frame())
display(pd.Series({"candidate_pool": len(candidates), "P75_screened_rules": len(screened),
    "Pareto_fronts": screened["pareto_front"].nunique(), "front_1_rules": int((fronts == 1).sum()),
    "unique_pairs_after_direction_selection": len(best_directions), "diversity_selected_rules": len(shortlist)}, name="Selection stages"))
with pd.option_context("display.max_rows", MAX_SHORTLIST_RULES, "display.max_colwidth", None):
    display(shortlist[["shortlist_rank", "pareto_front"] + display_columns + ["same_community", "selection_reason"]])

,P75 screening boundary
pair_support,0.007890
confidence,0.160367
lift,3.206209


,screened_rules
pareto_front,
1,3
2,5
3,8
4,10
5,18
6,18
7,22
8,24
9,20


candidate_pool                            9794
P75_screened_rules                        1255
Pareto_fronts                               50
front_1_rules                                3
unique_pairs_after_direction_selection     713
diversity_selected_rules                    20
Name: Selection stages, dtype: int64

,shortlist_rank,pareto_front,antecedent,consequent,antecedent_description,consequent_description,pair_count,pair_support,confidence,lift,same_community,selection_reason
0,1,1,22386,85099B,JUMBO BAG PINK POLKADOT,JUMBO BAG RED RETROSPOT,825,0.041724,0.677340,6.411222,True,"Front 1; support=4.17%, confidence=67.73%, lift=6.41 đạt P75; mỗi sản phẩm ≤ 2, cặp community ≤ 4."
1,2,1,22697,22699,GREEN REGENCY TEACUP AND SAUCER,ROSES REGENCY TEACUP AND SAUCER,767,0.038790,0.757157,14.057525,True,"Front 1; support=3.88%, confidence=75.72%, lift=14.06 đạt P75; mỗi sản phẩm ≤ 2, cặp community ≤ 4."
2,3,1,22698,22697,PINK REGENCY TEACUP AND SAUCER,GREEN REGENCY TEACUP AND SAUCER,632,0.031963,0.826144,16.125707,True,"Front 1; support=3.20%, confidence=82.61%, lift=16.13 đạt P75; mỗi sản phẩm ≤ 2, cặp community ≤ 4."
3,4,2,22698,22699,PINK REGENCY TEACUP AND SAUCER,ROSES REGENCY TEACUP AND SAUCER,598,0.030243,0.781699,14.513184,True,"Front 2; support=3.02%, confidence=78.17%, lift=14.51 đạt P75; mỗi sản phẩm ≤ 2, cặp community ≤ 4."
4,5,2,23300,23301,GARDENERS KNEELING PAD CUP OF TEA,GARDENERS KNEELING PAD KEEP CALM,546,0.027613,0.720317,15.600023,True,"Front 2; support=2.76%, confidence=72.03%, lift=15.60 đạt P75; mỗi sản phẩm ≤ 2, cặp community ≤ 4."
5,6,3,21931,85099B,JUMBO STORAGE BAG SUKI,JUMBO BAG RED RETROSPOT,724,0.036616,0.611486,5.787900,True,"Front 3; support=3.66%, confidence=61.15%, lift=5.79 đạt P75; mỗi sản phẩm ≤ 2, cặp community ≤ 4."
6,7,3,22383,20725,LUNCH BAG SUKI DESIGN,LUNCH BAG RED RETROSPOT,655,0.033126,0.510125,6.445172,True,"Front 3; support=3.31%, confidence=51.01%, lift=6.45 đạt P75; mỗi sản phẩm ≤ 2, cặp community ≤ 4."
7,8,3,22726,22727,ALARM CLOCK BAKELIKE GREEN,ALARM CLOCK BAKELIKE RED,640,0.032367,0.653061,12.286374,True,"Front 3; support=3.24%, confidence=65.31%, lift=12.29 đạt P75; mỗi sản phẩm ≤ 2, cặp community ≤ 4."
8,9,3,22910,22086,PAPER CHAIN KIT VINTAGE CHRISTMAS,PAPER CHAIN KIT 50'S CHRISTMAS,552,0.027917,0.673171,11.474659,True,"Front 3; support=2.79%, confidence=67.32%, lift=11.47 đạt P75; mỗi sản phẩm ≤ 2, cặp community ≤ 4."
9,10,3,22630,22629,DOLLY GIRL LUNCH BOX,SPACEBOY LUNCH BOX,538,0.027209,0.632941,13.905718,True,"Front 3; support=2.72%, confidence=63.29%, lift=13.91 đạt P75; mỗi sản phẩm ≤ 2, cặp community ≤ 4."


### Large invoices — shortlist

Dùng lại invoice_sets, large_invoice_ids và cache đếm cặp đã có; chỉ tính thêm các giao invoice sets chưa có trong cache. Hóa đơn lớn vẫn được xác định từ toàn bộ basket trước Top N. Large-invoice share không loại rule hoặc thay đổi ranking.

Đánh dấu large_invoice_caution cho share dương thuộc nhóm từ P75 của shortlist trở lên để ưu tiên kiểm tra khi diễn giải. Đây chỉ là mốc mô tả tương đối, không phải ngưỡng anomaly hay bằng chứng rule thiếu tin cậy.

Association metrics vẫn dùng baseline N đầy đủ. Không trừ riêng Pair Count rồi giữ nguyên mẫu số và single-product counts. Nếu thiếu community hợp lệ, metadata để trống, không áp dụng giới hạn cặp community; association và lựa chọn theo sản phẩm vẫn tiếp tục.

In [8]:
shortlist_keys = np.sort(shortlist[["antecedent", "consequent"]].to_numpy(), axis=1)
shortlist_large_counts = large_pair_counts.copy()
for a, b in shortlist_keys:
    if (a, b) not in shortlist_large_counts:
        shortlist_large_counts[(a, b)] = len(invoice_sets[a] & invoice_sets[b] & large_invoice_ids)
shortlist["large_invoice_pair_count"] = [shortlist_large_counts[(a, b)] for a, b in shortlist_keys]
shortlist["large_invoice_pair_share"] = shortlist["large_invoice_pair_count"] / shortlist["pair_count"]
large_share_p75 = shortlist["large_invoice_pair_share"].quantile(0.75)
shortlist["large_invoice_caution"] = shortlist["large_invoice_pair_share"].gt(0) & shortlist["large_invoice_pair_share"].ge(large_share_p75)
display(shortlist.sort_values(["large_invoice_pair_share", "shortlist_rank"], ascending=[False, True])[
    ["shortlist_rank", "antecedent", "consequent", "pair_count", "large_invoice_pair_count", "large_invoice_pair_share", "large_invoice_caution"]])


,shortlist_rank,antecedent,consequent,pair_count,large_invoice_pair_count,large_invoice_pair_share,large_invoice_caution
10,11,22356,20724,522,157,0.300766,True
1,2,22697,22699,767,217,0.282920,True
3,4,22698,22699,598,157,0.262542,True
2,3,22698,22697,632,164,0.259494,True
5,6,21931,85099B,724,176,0.243094,True
4,5,23300,23301,546,130,0.238095,False
15,16,22411,21931,524,121,0.230916,False
9,10,22630,22629,538,114,0.211896,False
8,9,22910,22086,552,116,0.210145,False
13,14,23202,23203,516,104,0.201550,False


## 7. Business Findings & Limitations

Các bảng bên dưới trích ví dụ từ kết quả hiện tại. Pair Support cho biết cặp thường xuất hiện đến đâu; Lift cho biết mức liên hệ so với tần suất riêng; Confidence xác định hướng gợi ý. Cùng Pair Count nhưng antecedent ít phổ biến hơn có thể tạo Confidence cao hơn, nên không suy ra A → B và B → A có cùng xác suất.

Những cặp cùng dòng, màu hoặc thiết kế có thể phản ánh sở thích nhất quán, mua nhiều biến thể hay mua sỉ. Chúng cũng có thể là sản phẩm thay thế, chưa chắc là bổ sung. Với các cặp khác loại, xem xét ngữ cảnh sử dụng từ Description và cấu trúc community trước khi thiết kế một giả thuyết bán kèm.

Có thể thử hiển thị consequent khi khách xem hoặc thêm antecedent vào giỏ, có kiểm soát sản phẩm sẵn hàng và ngữ cảnh. So sánh nhóm có/không có gợi ý bằng A/B testing, đo tỷ lệ mua thêm và giá trị giỏ hàng; không coi Confidence lịch sử là xác suất phản hồi của can thiệp.

Association không chứng minh nhân quả; Lift cao có thể đến từ cặp tương đối hiếm; Confidence phụ thuộc hướng; historical baskets chưa chứng minh tăng doanh thu. Community chỉ là bối cảnh. Large invoices có thể làm tăng co-purchase patterns. Phân tích chỉ bao phủ tập sản phẩm được chọn và chưa kiểm chứng trên thời gian/dữ liệu mới.

Nếu Lift > 1 giữ lại phần lớn rules, đó là bộ lọc ban đầu còn rộng. Cần đọc đồng thời Confidence, Support, số quan sát, ngữ cảnh bổ sung/thay thế và large-invoice share, không coi toàn bộ pool là shortlist kinh doanh. Xem hai hướng của cùng một cặp để chọn ngữ cảnh thử nghiệm; thứ hạng shortlist không bảo đảm hiệu quả triển khai.

Shortlist được chọn tự động từ association evidence. Tất cả quan hệ để unclear đến khi được business review. Những rule đầu bảng cung cấp các giả thuyết mua kèm có bằng chứng đồng xuất hiện mạnh; chưa xác nhận tính bổ sung, nhu cầu hay hiệu quả doanh thu.

Bảng đa dạng đối chiếu với bản shortlist thủ công trước thay đổi (mốc lịch sử 17 rules, không tham gia thuật toán). Các giới hạn giúp hạn chế lặp nhưng không bảo đảm mọi chỉ số đa dạng đều tăng; đọc cả số sản phẩm riêng biệt, tỷ lệ cùng/khác community và mức tập trung. Không bổ sung cross-community chỉ để tăng tỷ lệ.

Bảng Lift cao/Support thấp dùng median trong shortlist để mô tả mức tương đối, không bổ sung tiêu chí loại rule. So sánh với các cặp phổ biến hơn để tránh đồng nhất mức liên hệ tương đối với quy mô cơ hội. Hai nhóm cùng/khác community được đối chiếu trên metrics và Description, không coi nhãn community là điều kiện bán kèm.

In [9]:
if not shortlist.empty:
    business_examples = shortlist[["shortlist_rank"] + display_columns + ["relationship_type", "needs_business_review"]].copy()
    business_examples["reverse_confidence"] = business_examples["pair_count"] / business_examples["consequent"].map(single_counts)
    with pd.option_context("display.max_rows", 20, "display.max_colwidth", None):
        display(business_examples)
    community_summary = shortlist.groupby("same_community", dropna=False).agg(rules=("shortlist_rank", "size"), median_confidence=("confidence", "median"), median_pair_support=("pair_support", "median"), median_lift=("lift", "median"))
    community_summary["share"] = community_summary["rules"] / len(shortlist)
    display(community_summary)
    display(shortlist[["large_invoice_pair_share"]].agg(["min", "median", "max"]))
    display(shortlist.sort_values(["lift", "pair_support"], ascending=[False, True]).head(5)[["shortlist_rank"] + display_columns])
    lower_support_high_lift = shortlist.loc[shortlist["pair_support"].lt(shortlist["pair_support"].median()) & shortlist["lift"].gt(shortlist["lift"].median())]
    display(lower_support_high_lift[["shortlist_rank"] + display_columns].assign(comparison="Lift trên median nhưng Support dưới median shortlist"))
    display(shortlist.groupby("relationship_type").size().rename("rules"))
else:
    display(pd.Series({"shortlist_status": "No rule passes screening and diversity limits; no rules added to fill a quota."}))

# Historical summary of the manual shortlist; never used for selection.
manual_diversity = pd.Series({"rules": 17, "unique_products": 32, "unique_antecedents": 17,
    "unique_consequents": 15, "max_product_appearances": 2, "community_pairs": 6,
    "same_community_share": 15 / 17, "cross_community_share": 2 / 17}, name="Previous manual shortlist")
product_counts = pd.concat([shortlist["antecedent"], shortlist["consequent"]]).value_counts()
community_pair_counts = pd.DataFrame(np.sort(shortlist[["antecedent_community", "consequent_community"]].to_numpy(), axis=1)).value_counts() if community_available else pd.Series(dtype="int64")
automatic_diversity = pd.Series({"rules": len(shortlist), "unique_products": len(product_counts),
    "unique_antecedents": shortlist["antecedent"].nunique(), "unique_consequents": shortlist["consequent"].nunique(),
    "max_product_appearances": product_counts.max() if len(product_counts) else 0,
    "community_pairs": len(community_pair_counts) if community_available else np.nan,
    "same_community_share": shortlist["same_community"].eq(True).sum() / len(shortlist) if len(shortlist) and community_available else np.nan,
    "cross_community_share": shortlist["same_community"].eq(False).sum() / len(shortlist) if len(shortlist) and community_available else np.nan}, name="Automatic shortlist")
display(pd.concat([manual_diversity, automatic_diversity], axis=1))


,shortlist_rank,antecedent,consequent,antecedent_description,consequent_description,pair_count,pair_support,confidence,lift,relationship_type,needs_business_review,reverse_confidence
0,1,22386,85099B,JUMBO BAG PINK POLKADOT,JUMBO BAG RED RETROSPOT,825,0.041724,0.677340,6.411222,unclear,True,0.394926
1,2,22697,22699,GREEN REGENCY TEACUP AND SAUCER,ROSES REGENCY TEACUP AND SAUCER,767,0.038790,0.757157,14.057525,unclear,True,0.720188
2,3,22698,22697,PINK REGENCY TEACUP AND SAUCER,GREEN REGENCY TEACUP AND SAUCER,632,0.031963,0.826144,16.125707,unclear,True,0.623889
3,4,22698,22699,PINK REGENCY TEACUP AND SAUCER,ROSES REGENCY TEACUP AND SAUCER,598,0.030243,0.781699,14.513184,unclear,True,0.561502
4,5,23300,23301,GARDENERS KNEELING PAD CUP OF TEA,GARDENERS KNEELING PAD KEEP CALM,546,0.027613,0.720317,15.600023,unclear,True,0.598028
5,6,21931,85099B,JUMBO STORAGE BAG SUKI,JUMBO BAG RED RETROSPOT,724,0.036616,0.611486,5.787900,unclear,True,0.346577
6,7,22383,20725,LUNCH BAG SUKI DESIGN,LUNCH BAG RED RETROSPOT,655,0.033126,0.510125,6.445172,unclear,True,0.41853
7,8,22726,22727,ALARM CLOCK BAKELIKE GREEN,ALARM CLOCK BAKELIKE RED,640,0.032367,0.653061,12.286374,unclear,True,0.608944
8,9,22910,22086,PAPER CHAIN KIT VINTAGE CHRISTMAS,PAPER CHAIN KIT 50'S CHRISTMAS,552,0.027917,0.673171,11.474659,unclear,True,0.475862
9,10,22630,22629,DOLLY GIRL LUNCH BOX,SPACEBOY LUNCH BOX,538,0.027209,0.632941,13.905718,unclear,True,0.597778


,rules,median_confidence,median_pair_support,median_lift,share
same_community,,,,,
False,1,0.459016,0.025489,7.372974,0.05
True,19,0.611486,0.027613,11.292912,0.95


,large_invoice_pair_share
min,0.125000
median,0.193599
max,0.300766


,shortlist_rank,antecedent,consequent,antecedent_description,consequent_description,pair_count,pair_support,confidence,lift
2,3,22698,22697,PINK REGENCY TEACUP AND SAUCER,GREEN REGENCY TEACUP AND SAUCER,632,0.031963,0.826144,16.125707
4,5,23300,23301,GARDENERS KNEELING PAD CUP OF TEA,GARDENERS KNEELING PAD KEEP CALM,546,0.027613,0.720317,15.600023
3,4,22698,22699,PINK REGENCY TEACUP AND SAUCER,ROSES REGENCY TEACUP AND SAUCER,598,0.030243,0.781699,14.513184
1,2,22697,22699,GREEN REGENCY TEACUP AND SAUCER,ROSES REGENCY TEACUP AND SAUCER,767,0.038790,0.757157,14.057525
9,10,22630,22629,DOLLY GIRL LUNCH BOX,SPACEBOY LUNCH BOX,538,0.027209,0.632941,13.905718


,shortlist_rank,antecedent,consequent,antecedent_description,consequent_description,pair_count,pair_support,confidence,lift,comparison
9,10,22630,22629,DOLLY GIRL LUNCH BOX,SPACEBOY LUNCH BOX,538,0.027209,0.632941,13.905718,Lift trên median nhưng Support dưới median sho...
10,11,22356,20724,CHARLOTTE BAG PINK POLKADOT,RED RETROSPOT CHARLOTTE BAG,522,0.026400,0.703504,13.479056,Lift trên median nhưng Support dưới median sho...
14,15,22728,22727,ALARM CLOCK BAKELIKE PINK,ALARM CLOCK BAKELIKE RED,470,0.023770,0.600255,11.292912,Lift trên median nhưng Support dưới median sho...
17,18,85152,21166,HAND OVER THE CHOCOLATE SIGN,COOK WITH WINE METAL SIGN,295,0.014919,0.435746,11.917019,Lift trên median nhưng Support dưới median sho...


relationship_type
unclear    20
Name: rules, dtype: int64

,Previous manual shortlist,Automatic shortlist
rules,17.000000,20.00
unique_products,32.000000,29.00
unique_antecedents,17.000000,18.00
unique_consequents,15.000000,14.00
max_product_appearances,2.000000,2.00
community_pairs,6.000000,6.00
same_community_share,0.882353,0.95
cross_community_share,0.117647,0.05


## 8. Validation & Outputs

Giữ kiểm chứng trực tiếp tối đa 5 cặp mạnh từ pair_counts bằng invoice sets, công thức hai hướng và các validation association hiện có. Kiểm tra mỗi Pareto front đúng bằng 1 cộng front cao nhất trong các rules dominates nó (không có dominator thì bằng 1); các ties không dominates nhau.

Shortlist phải thuộc candidate pool, đạt P75, giữ nguyên metrics, không trùng directed/unordered pairs, đạt các giới hạn đa dạng và tối đa 20 rules. Kiểm tra large-invoice shares trong [0,1], lưu và đọc lại hai CSV. Nếu pool rỗng vẫn giữ schema, không hạ ngưỡng. Tính tái lập được kiểm tra bằng chạy lại notebook và so sánh output.

Giữ nguyên cross_selling_candidates.csv. Cập nhật cross_selling_shortlist.csv với metrics, pareto_front, shortlist_rank, community, business review và large-invoice share/caution. Không xuất thêm file trung gian hoặc thay đổi output Phase 2–3.

In [10]:
manual_rows = []
confidence_lookup = rules.set_index(["antecedent", "consequent"])["confidence"]
for a, b, pair_count in pairs.sort_values(["pair_count", "ProductA", "ProductB"], ascending=[False, True, True]).head(5).itertuples(index=False, name=None):
    count_a, count_b = len(invoice_sets[a]), len(invoice_sets[b])
    observed = len(invoice_sets[a] & invoice_sets[b])
    assert observed == pair_count
    if pair_count >= MIN_RULE_PAIR_COUNT:
        assert np.isclose(confidence_lookup.loc[(a, b)], observed / count_a) and np.isclose(confidence_lookup.loc[(b, a)], observed / count_b)
    manual_rows.append({"StockCodeA": a, "StockCodeB": b, "pair_count": observed, "support_a": count_a / N, "support_b": count_b / N,
        "pair_support": observed / N, "confidence_a_to_b": observed / count_a, "confidence_b_to_a": observed / count_b,
        "lift": observed * N / (count_a * count_b)})
manual_checks = pd.DataFrame(manual_rows)
if not pair_metrics.empty:
    verified = manual_checks.merge(pair_metrics, left_on=["StockCodeA", "StockCodeB"], right_on=["ProductA", "ProductB"], suffixes=("_direct", "_saved"))
    assert np.allclose(verified["pair_support_direct"], verified["pair_support_saved"]) and np.allclose(verified["lift_direct"], verified["lift_saved"])
display(manual_checks)
assert candidates[core_columns].notna().all().all() and np.isfinite(candidates[numeric_columns]).all().all()
assert candidates["pair_count"].ge(MIN_RULE_PAIR_COUNT).all() and candidates["lift"].gt(1).all()
output_path = TABLES_DIR / "cross_selling_candidates.csv"
candidates.to_csv(output_path, index=False)
saved = pd.read_csv(output_path, dtype={"antecedent": "string", "consequent": "string"})
pd.testing.assert_frame_equal(candidates.reset_index(drop=True), saved, check_dtype=False)
display(pd.Series({"positive_unordered_pairs": len(pairs), "eligible_unordered_pairs": len(pair_metrics), "directed_rules": len(rules),
    "exported_directed_candidates": len(candidates), "total_invoices_N": N, "community_integration": community_available,
    "output": str(output_path), "status": "saved and read back"}, name="Phase 4 summary"))


,StockCodeA,StockCodeB,pair_count,support_a,support_b,pair_support,confidence_a_to_b,confidence_b_to_a,lift
0,22386,85099B,825,0.061599,0.105649,0.041724,0.677340,0.394926,6.411222
1,22697,22699,767,0.051231,0.053861,0.038790,0.757157,0.720188,14.057525
2,21931,85099B,724,0.059880,0.105649,0.036616,0.611486,0.346577,5.787900
3,22411,85099B,680,0.059424,0.105649,0.034390,0.578723,0.325515,5.477787
4,20725,22383,655,0.079148,0.064937,0.033126,0.418530,0.510125,6.445172


positive_unordered_pairs                                                     4950
eligible_unordered_pairs                                                     4950
directed_rules                                                               9900
exported_directed_candidates                                                 9794
total_invoices_N                                                            19773
community_integration                                                        True
output                          /Users/trantuananh/co-purchase-network-finalte...
status                                                        saved and read back
Name: Phase 4 summary, dtype: object

In [11]:
assert len(shortlist) <= MAX_SHORTLIST_RULES and not shortlist.duplicated(["antecedent", "consequent"]).any()
assert not pd.DataFrame(shortlist_keys).duplicated().any()
assert shortlist[screen_metrics].ge(shortlist_bounds).all().all()
assert product_counts.le(MAX_PRODUCT_APPEARANCES).all()
assert community_pair_counts.le(MAX_COMMUNITY_PAIR_RULES).all()
assert np.array_equal(shortlist["pareto_front"], screened.set_index(["antecedent", "consequent"]).loc[pd.MultiIndex.from_frame(shortlist[["antecedent", "consequent"]]), "pareto_front"])
assert shortlist["large_invoice_pair_count"].between(0, shortlist["pair_count"]).all() and shortlist["large_invoice_pair_share"].between(0, 1).all()
pool_rows = saved.set_index(["antecedent", "consequent"])
shortlist_index = pd.MultiIndex.from_frame(shortlist[["antecedent", "consequent"]])
assert shortlist_index.isin(pool_rows.index).all()
pd.testing.assert_frame_equal(shortlist.set_index(["antecedent", "consequent"])[core_columns[2:]], pool_rows.loc[shortlist_index, core_columns[2:]], check_dtype=False)
shortlist_path = TABLES_DIR / "cross_selling_shortlist.csv"
shortlist.to_csv(shortlist_path, index=False)
saved_shortlist = pd.read_csv(shortlist_path, dtype={"antecedent": "string", "consequent": "string", "antecedent_community": "Int64", "consequent_community": "Int64", "same_community": "boolean"})
pd.testing.assert_frame_equal(shortlist.reset_index(drop=True), saved_shortlist, check_dtype=False)
display(pd.Series({"candidate_pool": len(candidates), "P75_screened_rules": len(screened), "Pareto_fronts": screened["pareto_front"].nunique(), "shortlisted_rules": len(shortlist),
    "same_community": int(shortlist["same_community"].eq(True).sum()), "cross_community": int(shortlist["same_community"].eq(False).sum()),
    "unknown_community": int(shortlist["same_community"].isna().sum()), "output": str(shortlist_path), "status": "saved and read back"}, name="Shortlist summary"))


candidate_pool                                                     9794
P75_screened_rules                                                 1255
Pareto_fronts                                                        50
shortlisted_rules                                                    20
same_community                                                       19
cross_community                                                       1
unknown_community                                                     0
output                /Users/trantuananh/co-purchase-network-finalte...
status                                              saved and read back
Name: Shortlist summary, dtype: object